# 01 - Instagram Public Data Scraping & Dual Format Output (CSV & XLSX)
Pipeline ekstraksi dan validasi data publik Instagram dari akun `@zahiraccounting` menggunakan Playwright (Async API) dengan penanganan akurat untuk `content_type`, collaborative posts, dan output ganda rapi berstandar CSV (RFC 4180) dan Excel (.xlsx).

## 1. Project Objective
Tujuan dari notebook ini adalah membangun pipeline web automation untuk mengekstrak seluruh postingan yang dapat diakses secara publik dalam batas sesi pengambilan data dari akun Instagram PT Zahir Internasional (`@zahiraccounting`). Pipeline ini dirancang untuk:
1. Mengklasifikasikan `content_type` secara presisi menggunakan badge indikator grid profil asli.
2. Menangani postingan kolaborasi (*collaborative posts*) secara terstruktur melalui variabel `is_collaboration` dan `original_account`.
3. Menjaga provenance URL dengan menyediakan canonical `post_url` dan `source_url`.
4. Mencegah kerusakan struktur tabel dengan menyediakan dua format penyimpanan terstandar:
   - **CSV**: Menggunakan `quoting=csv.QUOTE_ALL` berstandar RFC 4180 agar koma, quotes, dan newline di caption tidak memecah baris data.
   - **Excel (.xlsx)**: Menggunakan engine `openpyxl` dengan styling header elegan dan penyesuaian lebar kolom agar tampilan data di Microsoft Excel sangat rapi dan mudah dibaca langsung oleh manusia.

## 2. Data Source
- **Platform**: Instagram (Web)
- **Target Akun**: [@zahiraccounting](https://www.instagram.com/zahiraccounting/)
- **Karakteristik Data**: Postingan publik yang dapat diakses langsung oleh browser tanpa otentikasi/login.

## 3. Ethical and Privacy Considerations
1. **Strictly Public Data**: Hanya mengakses postingan yang terbuka untuk umum.
2. **Zero Credentials**: Tidak menyimpan username, password, token API, session, atau cookie.
3. **No Circumvention**: Tidak mem-bypass login wall, CAPTCHA, rate limit, atau mekanisme anti-bot Instagram.
4. **No Third-Party Services**: Tidak menggunakan layanan berbayar pihak ketiga seperti Apify.
5. **Data Privacy**: Tidak mengekstrak profil pribadi pengunjung atau komentator.
6. **No Dummy/Fabricated Data**: Nilai yang tidak tersedia secara publik (misalnya `views`) dibiarkan `NaN` tanpa dikarang.

## 4. Library Import and Environment Check
Memverifikasi environment Python 3.11, modul Playwright Async API, `openpyxl`, `csv`, dan `pandas`.

In [1]:
import sys
import os
import re
import csv
import json
import pandas as pd
import numpy as np
import openpyxl
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

print('Python Executable:', sys.executable)
print('Python Version:', sys.version)
print('Pandas Version:', pd.__version__)
print('OpenPyXL Version:', openpyxl.__version__)

Python Executable: D:\FILE KULIAH\semester 5\magang\zahir_social_analytics\.venv_social\Scripts\python.exe
Python Version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Pandas Version: 3.0.6
OpenPyXL Version: 3.1.5


## 5. Playwright Setup & Helper Functions
Mendefinisikan fungsi pembersih metrik numerik, helper penyimpanan Excel ber-styling rapi, dan konfigurasi browser context.

In [2]:
def clean_numeric_count(val):
    """Membersihkan string metrik likes/comments/views menjadi integer atau NaN."""
    if val is None or pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace(',', '')
    if 'k' in val_str.lower():
        return int(float(val_str.lower().replace('k', '')) * 1000)
    if 'm' in val_str.lower():
        return int(float(val_str.lower().replace('m', '')) * 1000000)
    try:
        return int(val_str)
    except (ValueError, TypeError):
        return np.nan

def save_styled_excel(df, excel_path, sheet_name='Sheet1'):
    """Menyimpan DataFrame ke file Excel (.xlsx) dengan styling rapi dan lebar kolom proporsional."""
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            df.to_excel(writer, index=False, sheet_name=sheet_name)
            ws = writer.sheets[sheet_name]
            header_fill = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')
            header_font = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
            thin_border = Border(
                left=Side(style='thin', color='D9D9D9'),
                right=Side(style='thin', color='D9D9D9'),
                top=Side(style='thin', color='D9D9D9'),
                bottom=Side(style='thin', color='D9D9D9')
            )
            for col_idx in range(1, len(df.columns) + 1):
                cell = ws.cell(row=1, column=col_idx)
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            for row in ws.iter_rows(min_row=2, max_row=len(df)+1, min_col=1, max_col=len(df.columns)):
                for cell in row:
                    cell.border = thin_border
                    cell.alignment = Alignment(vertical='top', wrap_text=False)
            for col in ws.columns:
                col_letter = col[0].column_letter
                col_name = str(col[0].value)
                if col_name == 'caption':
                    ws.column_dimensions[col_letter].width = 50
                elif 'url' in col_name:
                    ws.column_dimensions[col_letter].width = 38
                elif col_name in ['post_id', 'post_date', 'content_type', 'original_account']:
                    ws.column_dimensions[col_letter].width = 22
                elif col_name in ['hashtags']:
                    ws.column_dimensions[col_letter].width = 30
                else:
                    ws.column_dimensions[col_letter].width = 16
        print(f'Berhasil menyimpan file XLSX rapi ke {excel_path}')
    except PermissionError:
        print(f'Info: File {excel_path} sedang dibuka di Excel.')

TARGET_PROFILE_URL = 'https://www.instagram.com/zahiraccounting/'
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'

## 6. Instagram Public Page Access Test (Tahap 1A)
Menguji inisialisasi browser headless Chromium dan membuka profil publik `@zahiraccounting`.

In [3]:
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context(user_agent=USER_AGENT, locale='id-ID')
    page = await context.new_page()
    response = await page.goto(TARGET_PROFILE_URL, wait_until='networkidle', timeout=30000)
    
    print('HTTP Status:', response.status if response else 'None')
    print('Page Title:', await page.title())
    print('Current URL:', page.url)
    content = await page.content()
    print('HTML Content Length:', len(content))
    await browser.close()

HTTP Status: 200
Page Title: Zahir Accounting (@zahiraccounting) • Foto dan video Instagram
Current URL: https://www.instagram.com/zahiraccounting/
HTML Content Length: 871165


## 7. Post Discovery & Grid Badge Classification (Tahap 1B)
Mengambil seluruh tautan postingan yang dapat diakses dari grid profil publik. Penentuan `content_type` dilakukan langsung melalui badge grid profil (`Carousel`, `Clip` / `video/reel`, atau tanpa badge untuk `image`), sehingga terbebas dari bias elemen rekomendasi footer/rekomendasi postingan.

In [4]:
discovered_posts = []
seen_ids = set()

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context(user_agent=USER_AGENT, locale='id-ID')
    page = await context.new_page()
    await page.goto(TARGET_PROFILE_URL, wait_until='networkidle', timeout=30000)
    
    soup = BeautifulSoup(await page.content(), 'html.parser')
    anchors = soup.find_all('a')
    
    for a in anchors:
        href = a.get('href', '')
        if '/p/' in href or '/reel/' in href:
            match = re.search(r'/(?P<acc>[^/]+)/(?P<ptype>p|reel)/(?P<code>[^/]+)/', href)
            if not match:
                match = re.search(r'/(?P<ptype>p|reel)/(?P<code>[^/]+)/', href)
                acc = 'zahiraccounting'
            else:
                acc = match.group('acc')
                
            ptype = match.group('ptype')
            code = match.group('code')
            
            if code in seen_ids:
                continue
            seen_ids.add(code)
            
            # Klasifikasi content_type presisi dari grid badge
            a_text = a.get_text().strip().lower()
            svg_labels = [s.get('aria-label', '').lower() for s in a.find_all('svg') if s.get('aria-label')]
            all_labels = a_text + ' ' + ' '.join(svg_labels)
            
            if 'clip' in all_labels or ptype == 'reel':
                content_type = 'video/reel'
            elif 'carousel' in all_labels:
                content_type = 'carousel'
            else:
                content_type = 'image'
                
            canonical_url = f'https://www.instagram.com/{ptype}/{code}/'
            source_url = f'https://www.instagram.com/{acc}/{ptype}/{code}/'
            is_collab = (acc.lower() != 'zahiraccounting')
            orig_acc = acc.lower()
            
            discovered_posts.append({
                'post_id': code,
                'post_url': canonical_url,
                'source_url': source_url,
                'content_type': content_type,
                'is_collaboration': is_collab,
                'original_account': orig_acc
            })
            
    await browser.close()

df_discovered = pd.DataFrame(discovered_posts)
print(f'Total Postingan Publik Ditemukan: {len(df_discovered)}')
display(df_discovered)

Total Postingan Publik Ditemukan: 12


,post_id,post_url,source_url,content_type,is_collaboration,original_account
0,Dbr2KN3gZjt,https://www.instagram.com/p/Dbr2KN3gZjt/,https://www.instagram.com/zahiraccounting/p/Db...,carousel,False,zahiraccounting
1,DbGd0k0kuz4,https://www.instagram.com/p/DbGd0k0kuz4/,https://www.instagram.com/muhamad.is/p/DbGd0k0...,carousel,True,muhamad.is
2,DaKhtzOAeP1,https://www.instagram.com/p/DaKhtzOAeP1/,https://www.instagram.com/zahiraccounting/p/Da...,carousel,False,zahiraccounting
3,DaCQc-LEkwR,https://www.instagram.com/p/DaCQc-LEkwR/,https://www.instagram.com/muhamad.is/p/DaCQc-L...,carousel,True,muhamad.is
4,DZ7GcGdgVon,https://www.instagram.com/p/DZ7GcGdgVon/,https://www.instagram.com/zahiraccounting/p/DZ...,carousel,False,zahiraccounting
5,DY0oTVmEgxY,https://www.instagram.com/p/DY0oTVmEgxY/,https://www.instagram.com/zahiraccounting/p/DY...,image,False,zahiraccounting
6,DXJMBahiSxu,https://www.instagram.com/reel/DXJMBahiSxu/,https://www.instagram.com/modalngobrol/reel/DX...,video/reel,True,modalngobrol
7,DWIQL4qAWVe,https://www.instagram.com/p/DWIQL4qAWVe/,https://www.instagram.com/zahiraccounting/p/DW...,image,False,zahiraccounting
8,DV-ZtDBlKd4,https://www.instagram.com/p/DV-ZtDBlKd4/,https://www.instagram.com/zahiraccounting/p/DV...,image,False,zahiraccounting
9,DU7HlaWEgDx,https://www.instagram.com/p/DU7HlaWEgDx/,https://www.instagram.com/muhamad.is/p/DU7HlaW...,carousel,True,muhamad.is


## 8. Extraction Pipeline with Provenance & Collaboration Tracking (Tahap 1C)
Mengekstrak seluruh variabel postingan publik yang dapat diakses. Menangani teks caption, metrik likes/comments, tanggal rilis, status kolaborasi, dan tautan sumber.

In [5]:
extracted_records = []

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context(user_agent=USER_AGENT, locale='id-ID')
    page = await context.new_page()
    
    for idx, item in enumerate(discovered_posts, 1):
        url = item['post_url']
        print(f'[{idx}/{len(discovered_posts)}] Mengambil data: {url}')
        try:
            await page.goto(url, wait_until='domcontentloaded', timeout=20000)
            await page.wait_for_timeout(1000)
            post_soup = BeautifulSoup(await page.content(), 'html.parser')
            
            meta_desc = post_soup.find('meta', attrs={'name': 'description'})
            og_desc = post_soup.find('meta', attrs={'property': 'og:description'})
            desc_text = ''
            if meta_desc and meta_desc.get('content'):
                desc_text = meta_desc['content']
            elif og_desc and og_desc.get('content'):
                desc_text = og_desc['content']
                
            # Likes and Comments
            likes = np.nan
            comments = np.nan
            like_m = re.search(r'([0-9,.]+K?M?k?m?)\s+likes?', desc_text, re.IGNORECASE)
            if like_m:
                likes = clean_numeric_count(like_m.group(1))
            comm_m = re.search(r'([0-9,.]+K?M?k?m?)\s+comments?', desc_text, re.IGNORECASE)
            if comm_m:
                comments = clean_numeric_count(comm_m.group(1))
                
            # Caption parsing
            caption = None
            cap_m = re.search(r':\s*"(.*)"\s*\.?\s*$', desc_text, re.DOTALL)
            if cap_m:
                caption = cap_m.group(1).strip()
            elif ': "' in desc_text:
                parts = desc_text.split(': "', 1)
                if len(parts) > 1:
                    caption = parts[1].rstrip('". ')
            else:
                caption = desc_text if desc_text else None
                
            # Post Date
            time_el = post_soup.find('time')
            post_date = time_el.get('datetime') if time_el else None
            
            # Views (Tidak tersedia di publik web)
            views = np.nan
            
            # Hashtags & Caption Length
            if caption:
                ht_list = re.findall(r'#(\w+)', caption)
                hashtags_str = ' '.join(['#' + h for h in ht_list]) if ht_list else None
                ht_count = len(ht_list)
                caption_len = len(caption)
            else:
                hashtags_str = None
                ht_count = 0
                caption_len = 0
                
            extracted_records.append({
                'post_id': item['post_id'],
                'post_url': item['post_url'],
                'source_url': item['source_url'],
                'post_date': post_date,
                'content_type': item['content_type'],
                'caption': caption,
                'likes': likes,
                'comments': comments,
                'views': views,
                'hashtags': hashtags_str,
                'hashtag_count': ht_count,
                'caption_length': caption_len,
                'is_collaboration': item['is_collaboration'],
                'original_account': item['original_account']
            })
        except Exception as e:
            print(f'Error pada {url}: {e}')
            
    await browser.close()

df_posts = pd.DataFrame(extracted_records)
print(f'\nTotal postingan berhasil diekstrak: {len(df_posts)}')

[1/12] Mengambil data: https://www.instagram.com/p/Dbr2KN3gZjt/


[2/12] Mengambil data: https://www.instagram.com/p/DbGd0k0kuz4/


[3/12] Mengambil data: https://www.instagram.com/p/DaKhtzOAeP1/


[4/12] Mengambil data: https://www.instagram.com/p/DaCQc-LEkwR/


[5/12] Mengambil data: https://www.instagram.com/p/DZ7GcGdgVon/


...

[9/12] Mengambil data: https://www.instagram.com/p/DV-ZtDBlKd4/


[10/12] Mengambil data: https://www.instagram.com/p/DU7HlaWEgDx/


[11/12] Mengambil data: https://www.instagram.com/p/DThgjDGEt9O/


[12/12] Mengambil data: https://www.instagram.com/p/DS_l86HEqw7/



Total postingan berhasil diekstrak: 12


## 9. Comprehensive Data Validation
Melakukan validasi mendalam terhadap struktur data, distribusi `content_type`, kolom kolaborasi, missing values, dan verifikasi khusus postingan yang sebelumnya bermasalah (`DY0oTVmEgxY` dan `DWIQL4qAWVe`).

In [6]:
print('=== PREVIEW DATAFRAME (df_posts.head()) ===')
display(df_posts[['post_id', 'post_url', 'content_type', 'is_collaboration', 'original_account', 'likes', 'comments', 'views']].head())

print('\n=== DISTRIBUSI CONTENT_TYPE ===')
print(df_posts['content_type'].value_counts(dropna=False))

print('\n=== DISTRIBUSI IS_COLLABORATION ===')
print(df_posts['is_collaboration'].value_counts(dropna=False))

print('\n=== DISTRIBUSI ORIGINAL_ACCOUNT ===')
print(df_posts['original_account'].value_counts(dropna=False))

print('\n=== VALIDASI KHUSUS POSTINGAN YANG SEBELUMNYA BERMASALAH ===')
problem_check = df_posts[df_posts['post_id'].isin(['DY0oTVmEgxY', 'DWIQL4qAWVe'])][['post_id', 'content_type', 'is_collaboration', 'original_account']]
display(problem_check)

print('\n=== MISSING VALUES SUMMARY ===')
print(df_posts.isna().sum())

print('\n=== DUPLICATE POST_ID CHECK ===')
print('Jumlah duplikasi post_id:', df_posts.duplicated(subset=['post_id']).sum())

print('\n=== RENTANG TANGGAL POSTINGAN ===')
print('Postingan paling awal:', df_posts['post_date'].min())
print('Postingan paling akhir:', df_posts['post_date'].max())

=== PREVIEW DATAFRAME (df_posts.head()) ===


,post_id,post_url,content_type,is_collaboration,original_account,likes,comments,views
0,Dbr2KN3gZjt,https://www.instagram.com/p/Dbr2KN3gZjt/,carousel,False,zahiraccounting,4,0,NaN
1,DbGd0k0kuz4,https://www.instagram.com/p/DbGd0k0kuz4/,carousel,True,muhamad.is,34,2,NaN
2,DaKhtzOAeP1,https://www.instagram.com/p/DaKhtzOAeP1/,carousel,False,zahiraccounting,31,9,NaN
3,DaCQc-LEkwR,https://www.instagram.com/p/DaCQc-LEkwR/,carousel,True,muhamad.is,331,11,NaN
4,DZ7GcGdgVon,https://www.instagram.com/p/DZ7GcGdgVon/,carousel,False,zahiraccounting,13,12,NaN



=== DISTRIBUSI CONTENT_TYPE ===
content_type
carousel      8
image         3
video/reel    1
Name: count, dtype: int64

=== DISTRIBUSI IS_COLLABORATION ===
is_collaboration
False    6
True     6
Name: count, dtype: int64

=== DISTRIBUSI ORIGINAL_ACCOUNT ===
original_account
zahiraccounting    6
muhamad.is         4
modalngobrol       1
zahirerp           1
Name: count, dtype: int64

=== VALIDASI KHUSUS POSTINGAN YANG SEBELUMNYA BERMASALAH ===


,post_id,content_type,is_collaboration,original_account
5,DY0oTVmEgxY,image,False,zahiraccounting
7,DWIQL4qAWVe,image,False,zahiraccounting



=== MISSING VALUES SUMMARY ===
post_id              0
post_url             0
source_url           0
post_date            0
content_type         0
caption              0
likes                0
comments             0
views               12
hashtags             2
hashtag_count        0
caption_length       0
is_collaboration     0
original_account     0
dtype: int64

=== DUPLICATE POST_ID CHECK ===
Jumlah duplikasi post_id: 0

=== RENTANG TANGGAL POSTINGAN ===
Postingan paling awal: 2026-01-02T03:53:51.000Z
Postingan paling akhir: 2026-08-06T04:22:58.000Z


## 10. Save Output in Dual Format (CSV with QUOTE_ALL & Excel XLSX with OpenPyXL Styling)
Menyimpan dataset dalam **dua format terstandar**:
1. **CSV**: Menggunakan `quoting=csv.QUOTE_ALL` berstandar RFC 4180.
2. **Excel (.xlsx)**: Menggunakan engine `openpyxl` dengan styling header elegan dan lebar kolom yang disesuaikan.

In [7]:
output_test_csv = '../data/raw/instagram_test.csv'
output_raw_csv = '../data/raw/instagram_posts_raw.csv'
output_test_xlsx = '../data/raw/instagram_test.xlsx'
output_raw_xlsx = '../data/raw/instagram_posts_raw.xlsx'

# 1. Simpan CSV
try:
    df_posts.to_csv(output_test_csv, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL, lineterminator='\n')
    print(f'File CSV test berhasil disimpan ke {output_test_csv}')
except PermissionError:
    print(f'File {output_test_csv} sedang dibuka di aplikasi lain.')

try:
    df_posts.to_csv(output_raw_csv, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL, lineterminator='\n')
    print(f'File CSV raw berhasil disimpan ke {output_raw_csv}')
except PermissionError:
    print(f'File {output_raw_csv} sedang dibuka di aplikasi lain.')

# 2. Simpan Excel (.xlsx) dengan styling rapi
save_styled_excel(df_posts, output_test_xlsx, sheet_name='Instagram_Test')
save_styled_excel(df_posts, output_raw_xlsx, sheet_name='Instagram_Posts_Raw')

File ../data/raw/instagram_test.csv sedang dibuka di aplikasi lain.
File CSV raw berhasil disimpan ke ../data/raw/instagram_posts_raw.csv
Berhasil menyimpan file XLSX rapi ke ../data/raw/instagram_test.xlsx
Info: File ../data/raw/instagram_posts_raw.xlsx sedang dibuka di Excel.


## 11. Re-reading and Table Integrity Validation (CSV & XLSX)
Membaca ulang file CSV dan XLSX yang baru disimpan menggunakan `pd.read_csv()` dan `pd.read_excel()` untuk memverifikasi bahwa satu postingan tepat direpresentasikan oleh satu baris, kolom tidak bergeser, dan format tampilan rapi.

In [8]:
# Validasi pembacaan CSV
try:
    test_df_csv = pd.read_csv(output_test_csv, encoding='utf-8-sig')
    print('=== VALIDASI BACA ULANG CSV (test_df_csv) ===')
    print('Shape CSV:', test_df_csv.shape)
    print('Columns CSV:', test_df_csv.columns.tolist())
    display(test_df_csv.head())
except Exception as e:
    print('Info baca CSV:', e)

# Validasi pembacaan XLSX
try:
    test_df_xlsx = pd.read_excel(output_test_xlsx, sheet_name='Instagram_Test')
    print('\n=== VALIDASI BACA ULANG EXCEL (test_df_xlsx) ===')
    print('Shape XLSX:', test_df_xlsx.shape)
    print('Columns XLSX:', test_df_xlsx.columns.tolist())
    display(test_df_xlsx.head())
except Exception as e:
    print('Info baca XLSX:', e)

test_df = test_df_csv if 'test_df_csv' in locals() else test_df_xlsx

print('\n=== CEK STRUKTUR & TIPE DATA ===')
print('Rows:', len(test_df))
print('Columns:', len(test_df.columns))
print('\nDtypes:\n', test_df.dtypes)
print('\nMissing values:\n', test_df.isna().sum())

print('\n=== CEK CAPTION (3 CONTOH LENGKAP DENGAN NEWLINE ASLI) ===')
for i, caption in enumerate(test_df['caption'].head(3)):
    print(f'\n--- CAPTION {i+1} ---')
    print(caption)
    
print('\n=== CEK DUPLIKASI ===')
print('Duplicate post_id:', test_df['post_id'].duplicated().sum())
print('Duplicate post_url:', test_df['post_url'].duplicated().sum())

print('\n=== VALIDASI CONTENT_TYPE ===')
print(test_df['content_type'].value_counts(dropna=False))

print('\n=== VALIDASI KHUSUS DY0oTVmEgxY DAN DWIQL4qAWVe ===')
display(test_df[test_df['post_id'].isin(['DY0oTVmEgxY', 'DWIQL4qAWVe'])][['post_id', 'content_type', 'is_collaboration', 'original_account']])

=== VALIDASI BACA ULANG CSV (test_df_csv) ===
Shape CSV: (12, 14)
Columns CSV: ['post_id', 'post_url', 'source_url', 'post_date', 'content_type', 'caption', 'likes', 'comments', 'views', 'hashtags', 'hashtag_count', 'caption_length', 'is_collaboration', 'original_account']


,post_id,post_url,source_url,post_date,content_type,caption,likes,comments,views,hashtags,hashtag_count,caption_length,is_collaboration,original_account
0,Dbr2KN3gZjt,https://www.instagram.com/p/Dbr2KN3gZjt/,https://www.instagram.com/zahiraccounting/p/Db...,2026-08-06T04:22:58.000Z,carousel,Masih bingung menentukan harga jual produk bak...,4,0,NaN,#ZahirOnline #BisnisBakery #BelajarHPP #Manaje...,7,879,False,zahiraccounting
1,DbGd0k0kuz4,https://www.instagram.com/p/DbGd0k0kuz4/,https://www.instagram.com/muhamad.is/p/DbGd0k0...,2026-07-22T15:59:57.000Z,carousel,Dulu saya kira orang hebat adalah yang paling ...,34,2,NaN,#ERP #FinanceManager #Accounting #BusinessProc...,9,909,True,muhamad.is
2,DaKhtzOAeP1,https://www.instagram.com/p/DaKhtzOAeP1/,https://www.instagram.com/zahiraccounting/p/Da...,2026-06-29T09:18:04.000Z,carousel,"🚨 ""Fasilitas sih keren... tapi kok gak cuan, B...",31,9,NaN,#BisnisPadel #PadelIndonesia #ZahirAccounting ...,8,1132,False,zahiraccounting
3,DaCQc-LEkwR,https://www.instagram.com/p/DaCQc-LEkwR/,https://www.instagram.com/muhamad.is/p/DaCQc-L...,2026-06-26T04:13:13.000Z,carousel,Berapa banyak…\n\nBisnis yang dimulai bersama ...,331,11,NaN,NaN,0,685,True,muhamad.is
4,DZ7GcGdgVon,https://www.instagram.com/p/DZ7GcGdgVon/,https://www.instagram.com/zahiraccounting/p/DZ...,2026-06-23T09:54:39.000Z,carousel,"Omzet naik terus, tapi pas dicek kok saldonya ...",13,12,NaN,NaN,0,852,False,zahiraccounting



=== VALIDASI BACA ULANG EXCEL (test_df_xlsx) ===
Shape XLSX: (12, 14)
Columns XLSX: ['post_id', 'post_url', 'source_url', 'post_date', 'content_type', 'caption', 'likes', 'comments', 'views', 'hashtags', 'hashtag_count', 'caption_length', 'is_collaboration', 'original_account']


,post_id,post_url,source_url,post_date,content_type,caption,likes,comments,views,hashtags,hashtag_count,caption_length,is_collaboration,original_account
0,Dbr2KN3gZjt,https://www.instagram.com/p/Dbr2KN3gZjt/,https://www.instagram.com/zahiraccounting/p/Db...,2026-08-06T04:22:58.000Z,carousel,Masih bingung menentukan harga jual produk bak...,4,0,NaN,#ZahirOnline #BisnisBakery #BelajarHPP #Manaje...,7,879,False,zahiraccounting
1,DbGd0k0kuz4,https://www.instagram.com/p/DbGd0k0kuz4/,https://www.instagram.com/muhamad.is/p/DbGd0k0...,2026-07-22T15:59:57.000Z,carousel,Dulu saya kira orang hebat adalah yang paling ...,34,2,NaN,#ERP #FinanceManager #Accounting #BusinessProc...,9,909,True,muhamad.is
2,DaKhtzOAeP1,https://www.instagram.com/p/DaKhtzOAeP1/,https://www.instagram.com/zahiraccounting/p/Da...,2026-06-29T09:18:04.000Z,carousel,"🚨 ""Fasilitas sih keren... tapi kok gak cuan, B...",31,9,NaN,#BisnisPadel #PadelIndonesia #ZahirAccounting ...,8,1132,False,zahiraccounting
3,DaCQc-LEkwR,https://www.instagram.com/p/DaCQc-LEkwR/,https://www.instagram.com/muhamad.is/p/DaCQc-L...,2026-06-26T04:13:13.000Z,carousel,Berapa banyak…\n\nBisnis yang dimulai bersama ...,331,11,NaN,NaN,0,685,True,muhamad.is
4,DZ7GcGdgVon,https://www.instagram.com/p/DZ7GcGdgVon/,https://www.instagram.com/zahiraccounting/p/DZ...,2026-06-23T09:54:39.000Z,carousel,"Omzet naik terus, tapi pas dicek kok saldonya ...",13,12,NaN,NaN,0,852,False,zahiraccounting



=== CEK STRUKTUR & TIPE DATA ===
Rows: 12
Columns: 14

Dtypes:
 post_id                 str
post_url                str
source_url              str
post_date               str
content_type            str
caption                 str
likes                 int64
comments              int64
views               float64
hashtags                str
hashtag_count         int64
caption_length        int64
is_collaboration       bool
original_account        str
dtype: object

Missing values:
 post_id              0
post_url             0
source_url           0
post_date            0
content_type         0
caption              0
likes                0
comments             0
views               12
hashtags             2
hashtag_count        0
caption_length       0
is_collaboration     0
original_account     0
dtype: int64

=== CEK CAPTION (3 CONTOH LENGKAP DENGAN NEWLINE ASLI) ===

--- CAPTION 1 ---
Masih bingung menentukan harga jual produk bakery kamu? Atau sering kelabakan gara-gara stok baha

,post_id,content_type,is_collaboration,original_account
5,DY0oTVmEgxY,image,False,zahiraccounting
7,DWIQL4qAWVe,image,False,zahiraccounting


## 12. Fase 1.5 - Instagram Scraping Expansion & Public Coverage Audit
Pada fase ini dilakukan audit coverage dan eksplorasi ekspansi pengumpulan data secara publik tanpa login:
- **Discovery Feed**: Mengumpulkan 12 postingan awal pada grid feed profil.
- **Discovery Reels**: Memanfaatkan tab publik `/reels/` dengan akumulator scroll tanpa bypass login wall.
- **Public Access Limitation**: Instagram menerapkan login overlay pada scroll ke-8 di tab Reels dan menghentikan pagination feed pada 12 postingan pertama tanpa otentikasi.
- **Hasil Ekspansi**: Total 55 postingan publik unik berhasil diekstrak (12 retained dari Fase 1 + 43 postingan reels baru, 0 duplikasi).

In [ ]:
import pandas as pd
import numpy as np

# Membaca dataset hasil ekspansi Fase 1.5
df_expanded = pd.read_csv('../data/raw/instagram_posts_raw.csv', encoding='utf-8-sig')
print('=== HASIL EKSPANSI FASE 1.5 (df_expanded) ===')
print('Bentuk DataFrame (Rows, Cols):', df_expanded.shape)
print('Duplicate post_id:', df_expanded['post_id'].duplicated().sum())
print('Duplicate post_url:', df_expanded['post_url'].duplicated().sum())

print('\n=== DISTRIBUSI CONTENT_TYPE (FASE 1.5) ===')
print(df_expanded['content_type'].value_counts())

print('\n=== DISTRIBUSI KOLABORASI (FASE 1.5) ===')
print(df_expanded['is_collaboration'].value_counts())
print('\nDetail Akun Orisinal/Kolaborator:')
print(df_expanded['original_account'].value_counts())

print('\n=== RENTANG TANGGAL POSTINGAN ===')
print('Postingan paling awal:', df_expanded['post_date'].min())
print('Postingan paling akhir:', df_expanded['post_date'].max())

print('\n=== MISSING VALUES ===')
print(df_expanded.isna().sum())

## 13. Fase 1.6 & 1.7 - Discovery Expansion Audit, Sampling Bias, & Authenticated Access Feasibility

### Ringkasan Milestone Pengumpulan Data Instagram:
- **Phase 1**: 12 postingan publik awal (grid feed dasar).
- **Phase 1.5**: 55 postingan publik (ekspansi akumulator feed + reels tanpa login).
- **Phase 1.6**: Audit sampling bias dan penetapan **baseline publik terverifikasi (55 postingan)**. Ditemukan sampling bias teknis terhadap Reels (80%) karena batasan platform publik.
- **Phase 1.7**: Pengujian kelayakan akses terotentikasi (*Authenticated Access Feasibility Test*).

### Temuan Uji Kelayakan Otentikasi Manual (*Authenticated Observation*):
- Pada penjelajahan manual dengan peramban terotentikasi (*logged-in session*), postingan feed profil dapat digulir jauh melampaui 12 item (estimasi visibilitas: >500 postingan) dan reels melampaui 44 item (estimasi: >100 postingan), dengan arsip historis tertua mencapai tahun 2017.
- **Batasan Kebijakan & Keamanan Data Penting**:
  1. Observasi visibilitas manual (>500 postingan) **BUKAN BAGIAN DARI DATASET YANG DIKUMPULKAN**, melainkan observasi kelayakan teknis.
  2. Tidak ada kredensial pengguna, password, cookie, session token, atau token otentikasi pribadi yang diminta, disimpan, atau digunakan untuk otomasi scraping.
  3. Dataset aktif proyek **DIBEKUKAN PADA BASELINE SAH 55 POSTINGAN PUBLIK** (`data/raw/instagram_posts_raw.csv`) demi menjaga kepatuhan etika, keamanan sesi pengguna, dan standar integritas data akademik.

In [ ]:
# Verifikasi Baseline Terkunci (Frozen Baseline 55 Posts)
import pandas as pd

df_baseline = pd.read_csv('../data/raw/instagram_posts_raw.csv', encoding='utf-8-sig')
print('=== VERIFIKASI FROZEN BASELINE (PHASE 1.6 / 1.7) ===')
print('Total Rows (Posts):', len(df_baseline))
print('Total Columns:', len(df_baseline.columns))
print('Unique post_id:', df_baseline['post_id'].nunique())
print('Unique post_url:', df_baseline['post_url'].nunique())
print('Duplikasi post_id:', df_baseline['post_id'].duplicated().sum())

expected_columns = [
    'post_id', 'post_url', 'source_url', 'post_date', 'content_type',
    'caption', 'likes', 'comments', 'views', 'hashtags',
    'hashtag_count', 'caption_length', 'is_collaboration', 'original_account'
]
assert list(df_baseline.columns) == expected_columns, 'Schema mismatch!'
print('Schema 14-kolom valid sesuai standar proyek: 100% OK')

# Rekapitulasi Karakteristik Baseline Publik
print('\nDistribusi Format Konten Baseline Publik:')
print(df_baseline['content_type'].value_counts())
print('\nDistribusi Akun Orisinal / Kolaborasi:')
print(df_baseline['original_account'].value_counts())